Notebook 05 - Avaliação da LLM sem RAG

Baseline de Conhecimento Interno do Modelo

Este notebook avalia o desempenho do modelo de linguagem **Llama 3.1 8B Instruct**, acessado por meio da API do **OpenRouter**, na resolução de questões de múltipla escolha relacionadas a *Technology and Innovation Roadmapping*, sem o uso de Retrieval-Augmented Generation (RAG).

Diferentemente do sistema desenvolvido no Notebook 04, neste experimento o modelo não utiliza recuperação semântica, índice FAISS, embeddings ou trechos da base documental como contexto. As respostas são geradas exclusivamente a partir do conhecimento interno do modelo de linguagem.

O objetivo é estabelecer um **baseline de comparação** com o sistema RAG, permitindo avaliar o impacto da recuperação documental sobre o desempenho do modelo.

Para manter condições experimentais comparáveis, são utilizados o mesmo modelo de linguagem (**Llama 3.1 8B Instruct**), a mesma configuração de temperatura (`temperature = 0`), o mesmo formato estruturado de resposta e o mesmo conjunto de questões utilizado na avaliação do RAG.

Cada questão pode ser executada múltiplas vezes. As execuções são armazenadas separadamente, permitindo analisar tanto a acurácia das respostas quanto a consistência das alternativas selecionadas pelo modelo.

Preparação do Ambiente

In [ ]:
# Instala a biblioteca necessária para acesso ao OpenRouter via LangChain

!pip install -q -U langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.0/570.0 kB 13.4 MB/s eta 0:00:00


In [ ]:
# Importa as bibliotecas utilizadas no Notebook 05

import json
import re

from pathlib import Path
from datetime import datetime
from collections import Counter
from getpass import getpass

from google.colab import drive

from langchain_openai import ChatOpenAI

In [ ]:
# Monta o Google Drive

drive.mount(
    "/content/drive"
)

Mounted at /content/drive


In [ ]:
# Configura as pastas utilizadas pelo Notebook 05

BASE_DIR = Path(
    "/content/drive/MyDrive/RAG_Novo"
)

RESULTS_DIR = (
    BASE_DIR
    / "06_Resultados_LLM_Sem_RAG"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(
    "Pasta base:",
    BASE_DIR
)

print(
    "Pasta de resultados:",
    RESULTS_DIR
)

print(
    "Configuração das pastas concluída."
)

Pasta base: /content/drive/MyDrive/RAG_Novo
Pasta de resultados: /content/drive/MyDrive/RAG_Novo/06_Resultados_LLM_Sem_RAG
Configuração das pastas concluída.


Funções Auxiliares

In [ ]:
# Define funções auxiliares para padronizar a exibição das mensagens

def print_header(
    text
):
    print()
    print("=" * 70)
    print(
        f" {text}"
    )
    print("=" * 70)


def print_info(
    label,
    value
):
    print(
        f"{label:<25} {value}"
    )


def print_success(
    text
):
    print()
    print(
        f"✅ {text}"
    )


def print_warning(
    text
):
    print()
    print(
        f"⚠️ {text}"
    )


def print_error(
    text
):
    print()
    print(
        f"❌ {text}"
    )

Configuração do LLM

In [ ]:
# Configuração do modelo LLM utilizado no experimento

LLM_MODEL_NAME = "meta-llama/llama-3.1-8b-instruct"

print_header(
    "CONFIGURAÇÃO DO MODELO LLM"
)

print_info(
    "Provedor:",
    "OpenRouter"
)

print_info(
    "Modelo:",
    LLM_MODEL_NAME
)

print_info(
    "Modo:",
    "LLM sem RAG"
)

print_success(
    "Modelo configurado com sucesso."
)

print("=" * 70)


 CONFIGURAÇÃO DO MODELO LLM
Provedor:                 OpenRouter
Modelo:                   meta-llama/llama-3.1-8b-instruct
Modo:                     LLM sem RAG

✅ Modelo configurado com sucesso.


Configuração da API do OpenRouter

In [ ]:
# Configuração da API do OpenRouter

print_header(
    "CONFIGURAÇÃO DA API"
)

OPENROUTER_API_KEY = getpass(
    "Digite sua API Key do OpenRouter: "
)

if not OPENROUTER_API_KEY:

    raise ValueError(
        "A chave da API não foi informada."
    )

print_info(
    "Status da API:",
    "Configurada"
)

print_success(
    "API do OpenRouter configurada com sucesso."
)

print("=" * 70)


 CONFIGURAÇÃO DA API
Digite sua API Key do OpenRouter: ··········
Status da API:            Configurada

✅ API do OpenRouter configurada com sucesso.


Configuração do Cliente OpenRouter

In [ ]:
# Configura o cliente do OpenRouter via LangChain

print_header(
    "CONFIGURAÇÃO DO CLIENTE OPENROUTER"
)

if not OPENROUTER_API_KEY:

    raise ValueError(
        "A chave OPENROUTER_API_KEY não foi configurada."
    )

llm = ChatOpenAI(
    model=LLM_MODEL_NAME,
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1"
)

print_info(
    "Modelo:",
    LLM_MODEL_NAME
)

print_info(
    "Temperatura:",
    "Padrão do modelo/provedor (não definida manualmente)"
)

print_info(
    "Modo:",
    "Sem RAG"
)

print_success(
    "Cliente OpenRouter configurado com sucesso."
)

print("=" * 70)


 CONFIGURAÇÃO DO CLIENTE OPENROUTER
Modelo:                   meta-llama/llama-3.1-8b-instruct
Temperatura:              Padrão do modelo/provedor (não definida manualmente)
Modo:                     Sem RAG

✅ Cliente OpenRouter configurado com sucesso.


Definição do Prompt da LLM sem RAG

In [ ]:
# Define a instrução e o formato do prompt da LLM sem RAG

instruction = """
You are an AI assistant specialized in Technology and Innovation Roadmapping.

Your task is to answer multiple-choice questions using your internal knowledge only.

Carefully compare all alternatives A, B, C, D and E before selecting the answer.

Select exactly one alternative only when its complete meaning is considered correct.

Pay close attention to:

- the exact terminology used in the question;
- the order of concepts when the question asks for a sequence;
- distinctions between similar concepts;
- whether every element of an alternative is actually correct.

Do not select an alternative if it contains a term, relationship or ordering
that makes the alternative incorrect.

The value of "Answer" MUST reproduce exactly the full text of the selected
alternative as written in the question.

Provide a concise justification for the selected alternative.

Do not use retrieved documents, external context, or a RAG system.

If you cannot determine the correct alternative based on your internal knowledge,
state this clearly instead of guessing.

Return ONLY a valid JSON object using exactly this structure:

{
"Correct Alternative": "A, B, C, D or E",
"Answer": "Exact full text of the selected alternative",
"Justification": "Brief explanation for the selected alternative"
}

Do NOT include markdown.

Do NOT include text before or after the JSON.

Do NOT explain your reasoning process.

Return only the final JSON object.
""".strip()


llm_prompt_template = """### Instruction:

{}

### Input:

{}

### Response:

"""

In [ ]:
# Constrói o prompt completo enviado ao modelo

def build_prompt(
    question: str
):

    if not isinstance(question, str) or not question.strip():

        raise ValueError(
            "A pergunta não pode estar vazia."
        )

    prompt = llm_prompt_template.format(
        instruction,
        question.strip()
    )

    return prompt

Validação e Padronização da Resposta

In [ ]:
# Valida e padroniza a resposta estruturada retornada pela LLM
# Também recupera os campos quando o JSON retornado está levemente malformado.

def validate_llm_answer(
    parsed_answer,
    raw_answer
):

    # --------------------------------------------------
    # GARANTE QUE raw_answer SEJA TEXTO
    # --------------------------------------------------

    if raw_answer is None:
        raw_answer = ""

    raw_answer = str(
        raw_answer
    ).strip()

    # --------------------------------------------------
    # ESTRUTURA INICIAL
    # --------------------------------------------------

    if isinstance(
        parsed_answer,
        dict
    ):

        working_answer = dict(
            parsed_answer
        )

    else:

        working_answer = {}

    # --------------------------------------------------
    # RECUPERA A ALTERNATIVA
    # --------------------------------------------------

    correct_alternative = str(
        working_answer.get(
            "Correct Alternative",
            ""
        )
    ).strip().upper()

    # Se não veio corretamente no JSON,
    # tenta recuperar diretamente da resposta bruta.

    if correct_alternative not in {
        "A", "B", "C", "D", "E"
    }:

        alternative_match = re.search(
            r'"Correct Alternative"\s*:\s*"([A-E])"',
            raw_answer,
            flags=re.IGNORECASE
        )

        if alternative_match:

            correct_alternative = (
                alternative_match
                .group(1)
                .upper()
            )

        else:

            correct_alternative = ""

    # --------------------------------------------------
    # RECUPERA A RESPOSTA
    # --------------------------------------------------

    answer = working_answer.get(
        "Answer",
        ""
    )

    if answer is None:
        answer = ""

    answer = str(
        answer
    ).strip()

    # Se não foi recuperada pelo JSON,
    # tenta extrair da resposta bruta.

    if not answer:

        answer_match = re.search(
            r'"Answer"\s*:\s*"([^"]*)"',
            raw_answer,
            flags=re.IGNORECASE | re.DOTALL
        )

        if answer_match:

            answer = (
                answer_match
                .group(1)
                .strip()
            )

    # --------------------------------------------------
    # RECUPERA A JUSTIFICATIVA
    # --------------------------------------------------

    justification = working_answer.get(
        "Justification",
        ""
    )

    if justification is None:
        justification = ""

    justification = str(
        justification
    ).strip()

    # Em alguns erros de parsing,
    # raw_answer inteiro pode ter sido colocado
    # indevidamente em Justification.

    if (
        not justification
        or justification == raw_answer
        or '"Correct Alternative"' in justification
    ):

        justification = ""

        # Caso normal:
        # justificativa entre aspas e localizada
        # antes do fechamento do objeto JSON.

        justification_match = re.search(
            r'"Justification"\s*:\s*"(.+?)"\s*\}',
            raw_answer,
            flags=re.IGNORECASE | re.DOTALL
        )

        if justification_match:

            justification = (
                justification_match
                .group(1)
                .strip()
            )

        else:

            # Caso em que o modelo esquece as aspas
            # em volta da justificativa.

            justification_match = re.search(
                r'"Justification"\s*:\s*(.+?)\s*\}',
                raw_answer,
                flags=re.IGNORECASE | re.DOTALL
            )

            if justification_match:

                justification = (
                    justification_match
                    .group(1)
                    .strip()
                    .strip('"')
                )

    # --------------------------------------------------
    # RETORNO PADRONIZADO
    # --------------------------------------------------

    return {
        "Correct Alternative": correct_alternative,
        "Answer": answer,
        "Justification": justification
    }

Execução da Consulta sem RAG

In [ ]:
# Executa uma consulta diretamente na LLM, sem RAG

def llm_query(
    question: str
):

    # Validação da pergunta
    if not isinstance(question, str) or not question.strip():

        raise ValueError(
            "A pergunta não pode estar vazia."
        )

    # Verifica se o cliente foi inicializado
    if llm is None:

        raise RuntimeError(
            "O cliente OpenRouter não foi inicializado."
        )

    # Constrói o prompt
    full_prompt = build_prompt(
        question=question
    )

    # Envia a pergunta para a LLM
    try:

        response = llm.invoke(
            full_prompt
        )

    except Exception as error:

        raise RuntimeError(
            "Não foi possível obter uma resposta do modelo "
            "Llama 3.1 8B Instruct via OpenRouter. "
            "Verifique a conexão, a API Key, os créditos da conta "
            "e a disponibilidade do modelo."
        ) from error

    # Extrai a resposta textual
    raw_answer = str(
        response.content
    ).strip()

    if not raw_answer:

        raise RuntimeError(
            "O modelo retornou uma resposta vazia."
        )

    # --------------------------------------------------
    # TENTA INTERPRETAR O JSON NORMALMENTE
    # --------------------------------------------------

    try:

        parsed_answer = json.loads(
            raw_answer
        )

    except json.JSONDecodeError:

        # --------------------------------------------------
        # RECUPERAÇÃO DE RESPOSTA QUANDO O JSON
        # ESTÁ MALFORMADO, MAS OS CAMPOS ESTÃO PRESENTES
        # --------------------------------------------------

        correct_alternative = ""
        answer = ""
        justification = ""

        # Recupera a alternativa
        alternative_match = re.search(
            r'"Correct Alternative"\s*:\s*"([^"]*)"',
            raw_answer,
            flags=re.IGNORECASE
        )

        if alternative_match:

            correct_alternative = (
                alternative_match.group(1)
                .strip()
            )

        # Recupera a resposta
        answer_match = re.search(
            r'"Answer"\s*:\s*"([^"]*)"',
            raw_answer,
            flags=re.IGNORECASE
        )

        if answer_match:

            answer = (
                answer_match.group(1)
                .strip()
            )

        # Recupera a justificativa
        justification_match = re.search(
            r'"Justification"\s*:\s*"([^"]*)',
            raw_answer,
            flags=re.IGNORECASE | re.DOTALL
        )

        if justification_match:

            justification = (
                justification_match.group(1)
                .strip()
            )

            # Remove fechamento residual, se houver
            justification = re.sub(
                r'"\s*\}*\s*$',
                "",
                justification
            ).strip()

        parsed_answer = {
            "Correct Alternative": correct_alternative,
            "Answer": answer,
            "Justification": justification
        }

    # --------------------------------------------------
    # VALIDA E PADRONIZA
    # --------------------------------------------------

    parsed_answer = validate_llm_answer(
        parsed_answer=parsed_answer,
        raw_answer=raw_answer
    )

    # --------------------------------------------------
    # RETORNA OS ELEMENTOS DA CONSULTA
    # --------------------------------------------------

    return {
        "question": question.strip(),
        "correct_alternative": parsed_answer.get(
            "Correct Alternative",
            ""
        ),
        "answer": parsed_answer.get(
            "Answer",
            ""
        ),
        "justification": parsed_answer.get(
            "Justification",
            ""
        ),
        "raw_response": raw_answer,
        "prompt": full_prompt
    }

Salvamento dos Resultados

In [ ]:
# Salva cada pergunta em uma pasta própria.
# Cada nova execução da mesma pergunta gera um novo arquivo.
# Também atualiza automaticamente um arquivo de consistência.

def save_llm_result(
    llm_result,
    results_dir=RESULTS_DIR
):

    results_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    current_question = (
        llm_result["question"]
        .strip()
    )

    # --------------------------------------------------
    # 1. LOCALIZA OU CRIA A PASTA DA PERGUNTA
    # --------------------------------------------------

    question_dirs = sorted(
        [
            path
            for path in results_dir.iterdir()
            if path.is_dir()
            and re.fullmatch(
                r"\d+_pergunta_\d+",
                path.name
            )
        ]
    )

    question_dir = None
    question_number = None
    highest_question_number = 0

    for directory in question_dirs:

        match = re.fullmatch(
            r"(\d+)_pergunta_(\d+)",
            directory.name
        )

        if not match:
            continue

        folder_number = int(
            match.group(1)
        )

        highest_question_number = max(
            highest_question_number,
            folder_number
        )

        question_file = (
            directory
            / "pergunta.txt"
        )

        if question_file.exists():

            existing_question = (
                question_file
                .read_text(
                    encoding="utf-8"
                )
                .strip()
            )

            if existing_question == current_question:

                question_dir = directory
                question_number = folder_number

                break

    # Pergunta nova
    if question_dir is None:

        question_number = (
            highest_question_number + 1
        )

        question_dir = (
            results_dir
            / (
                f"{question_number:02d}_"
                f"pergunta_{question_number}"
            )
        )

        question_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        (
            question_dir
            / "pergunta.txt"
        ).write_text(
            current_question,
            encoding="utf-8"
        )

    # --------------------------------------------------
    # 2. DEFINE O NÚMERO DA NOVA EXECUÇÃO
    # --------------------------------------------------

    execution_files = [
        file_path
        for file_path in question_dir.glob(
            "execucao_*.txt"
        )
        if re.fullmatch(
            r"execucao_\d+\.txt",
            file_path.name
        )
    ]

    highest_execution_number = 0

    for file_path in execution_files:

        match = re.fullmatch(
            r"execucao_(\d+)\.txt",
            file_path.name
        )

        if match:

            highest_execution_number = max(
                highest_execution_number,
                int(
                    match.group(1)
                )
            )

    execution_number = (
        highest_execution_number + 1
    )

    output_file = (
        question_dir
        / f"execucao_{execution_number:02d}.txt"
    )

    # --------------------------------------------------
    # 3. MONTA O CONTEÚDO DA EXECUÇÃO
    # --------------------------------------------------

    lines = []

    lines.append(
        "=" * 80
    )

    lines.append(
        f"PERGUNTA Nº {question_number:02d} "
        f"- EXECUÇÃO Nº {execution_number:02d}"
    )

    lines.append(
        "=" * 80
    )

    lines.append("")

    lines.append(
        f"Data e hora: "
        f"{datetime.now().strftime('%d/%m/%Y %H:%M:%S')}"
    )

    lines.append(
        f"Modelo LLM: {LLM_MODEL_NAME}"
    )

    lines.append(
        "Modo: LLM sem RAG"
    )

    lines.append("")

    # Pergunta
    lines.append(
        "PERGUNTA"
    )

    lines.append(
        "-" * 80
    )

    lines.append(
        current_question
    )

    lines.append("")

    # Alternativa
    lines.append(
        "ALTERNATIVA CORRETA"
    )

    lines.append(
        "-" * 80
    )

    lines.append(
        str(
            llm_result.get(
                "correct_alternative",
                ""
            )
        )
    )

    lines.append("")

    # Resposta
    lines.append(
        "RESPOSTA"
    )

    lines.append(
        "-" * 80
    )

    lines.append(
        str(
            llm_result.get(
                "answer",
                ""
            )
        )
    )

    lines.append("")

    # Justificativa
    lines.append(
        "JUSTIFICATIVA"
    )

    lines.append(
        "-" * 80
    )

    lines.append(
        str(
            llm_result.get(
                "justification",
                ""
            )
        )
    )

    lines.append("")

    # Resposta bruta
    lines.append(
        "RESPOSTA BRUTA DO LLM"
    )

    lines.append(
        "-" * 80
    )

    lines.append(
        str(
            llm_result.get(
                "raw_response",
                ""
            )
        )
    )

    lines.append("")

    # Prompt completo
    lines.append(
        "PROMPT COMPLETO ENVIADO AO LLM"
    )

    lines.append(
        "-" * 80
    )

    lines.append(
        str(
            llm_result.get(
                "prompt",
                ""
            )
        )
    )

    lines.append("")

    lines.append(
        "=" * 80
    )

    output_file.write_text(
        "\n".join(lines),
        encoding="utf-8"
    )

    # --------------------------------------------------
    # 4. LOCALIZA TODAS AS EXECUÇÕES
    # --------------------------------------------------

    all_execution_files = [
        file_path
        for file_path in question_dir.glob(
            "execucao_*.txt"
        )
        if re.fullmatch(
            r"execucao_\d+\.txt",
            file_path.name
        )
    ]

    all_execution_files = sorted(
        all_execution_files,
        key=lambda file_path: int(
            re.fullmatch(
                r"execucao_(\d+)\.txt",
                file_path.name
            ).group(1)
        )
    )

    # --------------------------------------------------
    # 5. LÊ AS ALTERNATIVAS
    # --------------------------------------------------

    alternatives = []
    execution_results = []

    for file_path in all_execution_files:

        content = file_path.read_text(
            encoding="utf-8"
        )

        marker_start = (
            "ALTERNATIVA CORRETA\n"
            + "-" * 80
            + "\n"
        )

        marker_end = (
            "\n\nRESPOSTA"
        )

        alternative = ""

        if (
            marker_start in content
            and marker_end in content
        ):

            alternative = (
                content
                .split(
                    marker_start,
                    1
                )[1]
                .split(
                    marker_end,
                    1
                )[0]
                .strip()
                .upper()
            )

        if alternative in {
            "A", "B", "C", "D", "E"
        }:

            alternatives.append(
                alternative
            )

        execution_results.append(
            {
                "file": file_path.name,
                "alternative": alternative
            }
        )

    # --------------------------------------------------
    # 6. CALCULA A CONSISTÊNCIA
    # --------------------------------------------------

    most_common_alternative = ""
    consistency_percent = 0.0
    consistency_status = "Sem respostas válidas"

    if alternatives:

        counts = {
            alternative: alternatives.count(
                alternative
            )
            for alternative in {
                "A", "B", "C", "D", "E"
            }
        }

        max_count = max(
            counts.values()
        )

        predominant_alternatives = [
            alternative
            for alternative, count in counts.items()
            if count == max_count
            and count > 0
        ]

        consistency_percent = (
            max_count
            / len(alternatives)
            * 100
        )

        if len(
            predominant_alternatives
        ) == 1:

            most_common_alternative = (
                predominant_alternatives[0]
            )

            consistency_status = (
                "Alternativa predominante identificada"
            )

        else:

            most_common_alternative = (
                "EMPATE: "
                + ", ".join(
                    predominant_alternatives
                )
            )

            consistency_status = (
                "Empate entre alternativas"
            )

    # --------------------------------------------------
    # 7. SALVA O ARQUIVO DE CONSISTÊNCIA
    # --------------------------------------------------

    consistency_file = (
        question_dir
        / "consistencia.txt"
    )

    consistency_lines = []

    consistency_lines.append(
        "=" * 80
    )

    consistency_lines.append(
        f"CONSISTÊNCIA DA PERGUNTA Nº "
        f"{question_number:02d}"
    )

    consistency_lines.append(
        "=" * 80
    )

    consistency_lines.append("")

    consistency_lines.append(
        "PERGUNTA"
    )

    consistency_lines.append(
        "-" * 80
    )

    consistency_lines.append(
        current_question
    )

    consistency_lines.append("")

    consistency_lines.append(
        f"Total de execuções: "
        f"{len(all_execution_files)}"
    )

    consistency_lines.append(
        f"Execuções válidas: "
        f"{len(alternatives)}"
    )

    consistency_lines.append("")

    for item in execution_results:

        match = re.fullmatch(
            r"execucao_(\d+)\.txt",
            item["file"]
        )

        execution_id = (
            int(match.group(1))
            if match
            else 0
        )

        consistency_lines.append(
            f"Execução {execution_id:02d}: "
            f"{item['alternative']}"
        )

    consistency_lines.append("")

    consistency_lines.append(
        f"Alternativa predominante: "
        f"{most_common_alternative}"
    )

    consistency_lines.append(
        f"Consistência: "
        f"{consistency_percent:.2f}%"
    )

    consistency_lines.append(
        f"Status: "
        f"{consistency_status}"
    )

    consistency_lines.append("")

    consistency_lines.append(
        "=" * 80
    )

    consistency_file.write_text(
        "\n".join(
            consistency_lines
        ),
        encoding="utf-8"
    )

    # --------------------------------------------------
    # 8. RETORNA INFORMAÇÕES DO SALVAMENTO
    # --------------------------------------------------

    return {
        "file": output_file,
        "question_dir": question_dir,
        "question_number": question_number,
        "execution_number": execution_number,
        "consistency_file": consistency_file,
        "most_common_alternative": most_common_alternative,
        "consistency_percent": consistency_percent,
        "consistency_status": consistency_status,
        "total_executions": len(
            all_execution_files
        ),
        "valid_executions": len(
            alternatives
        )
    }

Verificação Final do Notebook 05

In [ ]:
# Realiza a verificação final do Notebook 05

print_header(
    "VERIFICAÇÃO FINAL DO NOTEBOOK 05"
)

checks = {
    "Cliente OpenRouter": (
        llm is not None
    ),
    "Modelo LLM": (
        isinstance(
            LLM_MODEL_NAME,
            str
        )
        and bool(
            LLM_MODEL_NAME.strip()
        )
    ),
    "Construção do prompt": (
        callable(
            build_prompt
        )
    ),
    "Validação da resposta": (
        callable(
            validate_llm_answer
        )
    ),
    "Consulta sem RAG": (
        callable(
            llm_query
        )
    ),
    "Salvamento dos resultados": (
        callable(
            save_llm_result
        )
    ),
    "Pasta de resultados": (
        RESULTS_DIR.exists()
        and RESULTS_DIR.is_dir()
    )
}

failed_checks = []

for name, status in checks.items():

    print_info(
        f"{name}:",
        "OK" if status else "ERRO"
    )

    if not status:

        failed_checks.append(
            name
        )

print()

print_info(
    "Modelo LLM:",
    LLM_MODEL_NAME
)

print_info(
    "Temperatura:",
    "Padrão do modelo/provedor (não definida manualmente)"
)

print_info(
    "Modo experimental:",
    "LLM sem RAG"
)

print_info(
    "Pasta de resultados:",
    RESULTS_DIR
)

print_info(
    "Estrutura de salvamento:",
    "Pasta por pergunta + arquivo por execução"
)

print_info(
    "Arquivo de consistência:",
    "Atualizado automaticamente"
)

if failed_checks:

    print()

    print_error(
        "Existem componentes que precisam ser revisados."
    )

    print()

    print(
        "Componentes com problema:"
    )

    for item in failed_checks:

        print(
            f"• {item}"
        )

    raise RuntimeError(
        "A verificação final do Notebook 05 encontrou erros."
    )

print_success(
    "Notebook 05 configurado e validado com sucesso."
)

print("=" * 70)


 VERIFICAÇÃO FINAL DO NOTEBOOK 05
Cliente OpenRouter:       OK
Modelo LLM:               OK
Construção do prompt:     OK
Validação da resposta:    OK
Consulta sem RAG:         OK
Salvamento dos resultados: OK
Pasta de resultados:      OK

Modelo LLM:               meta-llama/llama-3.1-8b-instruct
Temperatura:              Padrão do modelo/provedor (não definida manualmente)
Modo experimental:        LLM sem RAG
Pasta de resultados:      /content/drive/MyDrive/RAG_Novo/06_Resultados_LLM_Sem_RAG
Estrutura de salvamento:  Pasta por pergunta + arquivo por execução
Arquivo de consistência:  Atualizado automaticamente

✅ Notebook 05 configurado e validado com sucesso.


Interface de Consulta da LLM sem RAG

In [ ]:
# Interface automática para executar o questionário completo diretamente na LLM
# Cada questão é enviada em uma chamada independente.
# Não há recuperação de documentos nem utilização de RAG.
# A sequência Q1 -> Q20 é repetida pelo número de rodadas definido.

# ============================================================
# CONFIGURAÇÃO DO EXPERIMENTO
# ============================================================

NUM_RODADAS = 10

# Para alterar futuramente:
# NUM_RODADAS = 20
# NUM_RODADAS = 30
# etc.


# ============================================================
# QUESTIONÁRIO
# ============================================================

QUESTIONS = [

    """
Technology Roadmapping started in companies, not in the academy.
What company was a pioneer in roadmapping and is mentioned in an academic paper published in the 80's?
A) Philips
B) Motorola
C) General Motors
D) Lucent Technologies
E) Nasa
""",

    """
The roadmap layers can be organized to support diffent sectors.
What have been the most used roadmap layers, starting from the top to the bottom, proposed in the academy
to create a fast-start roadmapping approach?
A) Innovation, Risk, Value
B) Technology, Market, Business
C) Knowledge, Culture, Strategy
D) Market, Product, Technology
E) Finance, Operations, Sales
""",

    """
What is the core motivation to apply the roadmapping approach?
A) Benchmark competitors working with similar technologies
B) Develop a strategic plan for products and technologies
C) Develop a business plan focused on technological innovation
D) Replace project management tools with updated gant charts and product roadmaps
E) Map the business trajectory considering different scenarios
""",

    """
What factors can be considered the most important when defining the roadmap timeline:
A) Industrial sector and business strategy
B) Market share and dynamics
C) Government regulation and intellectual property
D) Company and market size
E) Leadership and cultural approach
""",

    """
Roadmapping processes can provide tangible and intangible outcomes.
What is the main tangible output of a roadmapping process:
A) Business case integrated with a technological plan
B) Product plan communication in different departments
C) Strategic roadmap describing product and technological goals
D) Alignment and consensus among the roadmapping stakeholders
E) Sharing of knowledge developed through roadmapping
""",

    """
What are the main two approaches used in practice to support roadmapping applications?
A) Expert and data-driven roadmapping
B) Expert and project-based roadmapping
C) Strategic and product-based roadmapping
D) Computer-based and data-driven roadmapping
E) Vectorial and tabular-based roadmapping
""",

    """
What are the two innovation and technology tools indicated in the alternatives that can help with risk mitigation in roadmapping?
A) Quality function deployment and functional modeling
B) Scenario planning and portfolio management
C) Linking grids and morphological matrices
D) SWOT and Porter's 5 forces
E) Technology readiness level (TRL) and gantt charts
""",

    """
What alternative is NOT a barrier to the application of roadmapping?
A) Lack of strategic information during the early stages
B) Fast-changing customer requirements related to uncertain environments
C) Uncertainties in technology maturity and ecosystems
D) Application of product features and platforms as strategic orientation
E) Dynamic markets and competition when applied to small and medium companies
""",

    """
The expert-based roadmapping has been the most used approach over the years.
What technique does it adopt as central for roadmapping development?
A) Market analysis and trend modeling support by experts
B) Computational forecasting integrated with topic modeling
C) Collaborative and multidisciplinary workshops
D) Long-term projects created by specialized engineering teams
E) Teams focused on the technologies required for the roadmap
""",

    """
Several research institutes have supported roadmapping over time, forming the roadmapping schools of thought.
What is the academic institution most acknowledged for its contribution to roadmapping, with influence on all other schools?
A) Portland State University
B) University of Sao Paulo
C) University of Cambridge
D) National Seoul University
E) Northwestern University
""",

    """
Roadmapping and roadmaps refers to different parts of a roadmapping approach.
A roadmap can be best defined as:
A) A map that presents the entire story of business development
B) A guide to support strategic decisions in dynamic and uncertain technological domains
C) A chart that describes the actions planned for the company in the long-term.
D) A vision of the goals to be achieved by a business developed by a group of experts.
E) A layered and temporal graphical representation of strategic options to a company pursue
regarding innovation and technology development.
""",

    """
Despite some resistance, recent studies started using digital technologies for roadmapping.
What alternative present advantages noted in a digital roadmapping approach:
A) Better management of technology uncertainty and use of digital communication
B) Facilitated collaboration from different locations and faster information processing
C) Development of multiple roadmapping types and use of roadmap software
D) Ensure better engagement of experts in the process because of conference meetings and video sharing.
E) Support better collection of information for the technological layer and improved experts' discussion.
""",

    """
The application of scenario planning integrated with roadmapping is a common practice to improve roadmapping results.
What alternative indicates a benefit that scenario planning brings to roadmapping:
A) It helps to better address the long-term and vision strategies
B) It helps to better connect products and technologies' interdependencies
C) It helps to reduce the lead time to reach a qualified roadmap result
D) It helps to organize the roadmapping workshops and clarify the expected results in uncertain markets
E) It helps to assess the maturity of the existing technologies in multiple marks.
""",

    """
The development of an expert-based roadmapping involves multidisciplinary teams that need support to reach the expected result.
What is the alternative that presents a relevant role in this type of roadmapping to ensure collaborative work?
A) Workshop facilitator
B) Technical consultant
C) Business manager
D) Engineering expert
E) Marketing expert
""",

    """
There are several quantitative techniques used for computer or data-driven roadmapping.
These techniques have used data from several sources, but the main source used for this type of roadmapping is:
A) Social networks data
B) Patent data
C) Industrial reports data
D) Marketing data
E) Quantitative optimization data
""",

    """
What is NOT suggested as a practice to improve roadmapping performance and results?
A) Development of a pilot roadmapping project in companies with little experience
B) Define clear roadmapping goals and outcome expectations
C) Define a clear and feasible unit of analysis embracing the main product features
D) The involvement of experts with a technical and commercial background
E) Address roadmapping as a one-time application, without considering relevant values for the implementation of its result
""",

    """
The connection between roadmap layers is a relevant practice, in particular for product-technology roadmaps.
What tool is especially relevant for connecting roadmapping layers?
A) Linking grids
B) Technology Re-Linking Levels (TRL)
C) STEEP and SWOT
D) Scenario Planning
E) Decision Criteria
""",

    """
The roadmap layout is built upon a structure designed to organize information in layers and a timeline.
What are the three questions used to guide the collection of information that needs to be organized through the roadmap layers?
A) Where are we now? Where do we want to go? How do we get there?
B) Why do we need it? What do we need? How do we develop it?
C) Who are our customers? What do they buy? How do we contact them?
D) What are our strategies? How do we implement i? How do we measure results?
E) What is our vision? What is our current situation? What are our plans?
""",

    """
The roadmapping approach can be defined as:
A) A management approach used to map and manage product projects successfully launched in the market
B) A tool that supports improved decision-making based on business strategies and marketing goals
C) A management tool used to support technical departments in deciding what technology to develop
D) As approach that aims to develop a roadmap containing business directions prioritized for different market segments
E) A management approach used to identify, define, and map innovation strategies, objectives, and actions within a business or organization
""",

    """
The main outcome of roadmapping, the roadmap, can be also used as a diagnostic tool.
How the roadmap visual features can be used to guide roadmapping improvements:
A) They can indicate areas in which the roadmapping process is missing information or provides insufficient results
B) They can report with figures if the roadmapping process was conducted with commitment
C) They can show if the roadmapping experts were chosen correctly to provide the required information
D) They can present visually the business priorities that need to be considered in strategic actions
E) They can use symbols and signals to help communicate results and support organizational adoption
"""
]


# ============================================================
# VALIDAÇÕES INICIAIS
# ============================================================

if NUM_RODADAS <= 0:

    raise ValueError(
        "NUM_RODADAS deve ser maior que zero."
    )

if len(QUESTIONS) != 20:

    raise ValueError(
        f"O questionário deveria conter 20 questões, "
        f"mas contém {len(QUESTIONS)}."
    )


# ============================================================
# INÍCIO DO EXPERIMENTO
# ============================================================

total_questions = len(QUESTIONS)

total_expected_calls = (
    total_questions
    * NUM_RODADAS
)

successful_calls = 0
failed_calls = 0

failed_executions = []


print_header(
    "EXECUÇÃO AUTOMÁTICA DO QUESTIONÁRIO - LLM SEM RAG"
)

print_info(
    "Número de questões:",
    total_questions
)

print_info(
    "Número de rodadas:",
    NUM_RODADAS
)

print_info(
    "Total previsto de chamadas:",
    total_expected_calls
)

print_info(
    "Modelo LLM:",
    LLM_MODEL_NAME
)

print_info(
    "Temperatura:",
    "Padrão do modelo/provedor (não definida manualmente)"
)

print_info(
    "Modo experimental:",
    "LLM sem RAG"
)

print()

print_warning(
    "Cada questão será enviada em uma chamada independente à LLM."
)

print("=" * 70)


# ============================================================
# EXECUÇÃO DAS RODADAS
# ============================================================

for round_number in range(
    1,
    NUM_RODADAS + 1
):

    print()

    print_header(
        f"RODADA {round_number:02d} DE {NUM_RODADAS:02d}"
    )

    for question_number, question in enumerate(
        QUESTIONS,
        start=1
    ):

        print()

        print(
            f"[Rodada {round_number:02d}/{NUM_RODADAS:02d}] "
            f"Processando questão "
            f"{question_number:02d}/{total_questions:02d}..."
        )

        try:

            # ----------------------------------------------
            # CHAMADA INDEPENDENTE À LLM SEM RAG
            # ----------------------------------------------

            result = llm_query(
                question=question.strip()
            )

            # ----------------------------------------------
            # SALVAMENTO INDIVIDUAL DA EXECUÇÃO
            # ----------------------------------------------

            save_info = save_llm_result(
                result
            )

            successful_calls += 1

            print_info(
                "Alternativa:",
                (
                    result["correct_alternative"]
                    if result["correct_alternative"]
                    else "Não identificada"
                )
            )

            print_info(
                "Arquivo:",
                save_info["file"].name
            )

            print_info(
                "Consistência atual:",
                f"{save_info['consistency_percent']:.2f}%"
            )

            print_success(
                f"Questão {question_number:02d} concluída e salva."
            )

        except Exception as error:

            failed_calls += 1

            failed_executions.append(
                {
                    "round": round_number,
                    "question": question_number,
                    "error": str(error)
                }
            )

            print_error(
                f"Erro na questão {question_number:02d} "
                f"da rodada {round_number:02d}."
            )

            print(
                str(error)
            )

            print_warning(
                "A execução continuará para a próxima questão."
            )


# ============================================================
# RESUMO FINAL
# ============================================================

print()

print_header(
    "EXPERIMENTO CONCLUÍDO - LLM SEM RAG"
)

print_info(
    "Rodadas solicitadas:",
    NUM_RODADAS
)

print_info(
    "Questões por rodada:",
    total_questions
)

print_info(
    "Chamadas previstas:",
    total_expected_calls
)

print_info(
    "Chamadas concluídas:",
    successful_calls
)

print_info(
    "Chamadas com erro:",
    failed_calls
)

print_info(
    "Pasta de resultados:",
    RESULTS_DIR
)

print()

if failed_calls == 0:

    print_success(
        "Todas as chamadas foram concluídas e salvas com sucesso."
    )

else:

    print_warning(
        "O experimento terminou, mas algumas chamadas apresentaram erro."
    )

    print()

    print(
        "Execuções com erro:"
    )

    for failure in failed_executions:

        print(
            f"• Rodada {failure['round']:02d} | "
            f"Questão {failure['question']:02d} | "
            f"{failure['error']}"
        )

print("=" * 70)


 EXECUÇÃO AUTOMÁTICA DO QUESTIONÁRIO - LLM SEM RAG
Número de questões:       20
Número de rodadas:        10
Total previsto de chamadas: 200
Modelo LLM:               meta-llama/llama-3.1-8b-instruct
Temperatura:              Padrão do modelo/provedor (não definida manualmente)
Modo experimental:        LLM sem RAG


⚠️ Cada questão será enviada em uma chamada independente à LLM.


 RODADA 01 DE 10

[Rodada 01/10] Processando questão 01/20...
Alternativa:              A
Arquivo:                  execucao_01.txt
Consistência atual:       100.00%

✅ Questão 01 concluída e salva.

[Rodada 01/10] Processando questão 02/20...
Alternativa:              B
Arquivo:                  execucao_01.txt
Consistência atual:       100.00%

✅ Questão 02 concluída e salva.

[Rodada 01/10] Processando questão 03/20...
Alternativa:              B
Arquivo:                  execucao_01.txt
Consistência atual:       100.00%

✅ Questão 03 concluída e salva.

[Rodada 01/10] Processando questão 04/20...
Altern